# The AI-powered Entity Audit — run the course scripts in Colab

No installation needed. Run the **Setup** cell first, then jump to the section for your lesson.

**Before you start:** click the 🔑 key icon in the left sidebar → *Add new secret*, and add the keys your script needs, with *Notebook access* switched on:
- `ANTHROPIC_API_KEY` — [console.anthropic.com](https://console.anthropic.com/) → API Keys (Practicals 5.2, 5.3 with rewrites, 5.6)
- `DATAFORSEO_LOGIN` and `DATAFORSEO_PASSWORD` — [app.dataforseo.com/api-access](https://app.dataforseo.com/api-access) (Practical 4.3)

To run a cell, click the ▶ button on its left (or press Shift+Enter). Wait for the green tick before running the next one.

## Setup — run this first

In [ ]:
import os, shutil
if not os.path.exists('mlforseo-code'):
    !git clone -q --depth 1 https://github.com/lazarinastoy/mlforseo-code.git
!pip -q install -r mlforseo-code/the-ai-powered-entity-audit/requirements.txt

from google.colab import userdata, files
for name in ['ANTHROPIC_API_KEY', 'DATAFORSEO_LOGIN', 'DATAFORSEO_PASSWORD']:
    try:
        os.environ[name] = userdata.get(name)
        print('✓', name, 'loaded')
    except Exception:
        print('–', name, 'not set (only needed for some scripts)')

WORK = '/content/work'
os.makedirs(WORK, exist_ok=True)
for sub in ['module-4', 'module-5']:
    for f in os.listdir(f'mlforseo-code/the-ai-powered-entity-audit/{sub}'):
        src = f'mlforseo-code/the-ai-powered-entity-audit/{sub}/{f}'
        if os.path.isfile(src):
            shutil.copy(src, WORK)
%cd /content/work
print('Ready. Your files go into /content/work (the folder icon on the left).')

---
## Practical 4.3 — AI-citation analysis (DataForSEO)

1. Edit the topics below — one per line, the way people ask an assistant.
2. Run the **plan** cell: it prices the run and sends nothing.
3. Run the **run** cell, then open `samples.csv`, mark `on_topic` yes/no, upload it back, and run **report**.

In [ ]:
topics = '''entity seo
knowledge graph optimisation
'''
open('topics.txt', 'w').write(topics)
YOUR_DOMAIN = 'yoursite.com'      # without https:// or www
KNOWN_QUERY = 'entity seo'          # a topic whose cited sites you can name
!python ai_citation_queries.py plan topics.txt --domain {YOUR_DOMAIN} --known "{KNOWN_QUERY}" --location-code 2826 --language-code en

In [ ]:
!python ai_citation_queries.py run
files.download('samples.csv')

In [ ]:
print('Upload samples.csv after you have filled the on_topic column')
files.upload()
!python ai_citation_queries.py report --brand "Your Brand"
files.download('report.md')

---
## Practical 4.4 — strip boilerplate from competitor corpora (free)

Upload each site's `internal_all.csv` (and `entity_mentions.csv` if you have it). The cell puts each upload into a folder named after the site.

In [ ]:
SITE = 'competitor-one'   # change for each site, then run again
os.makedirs(SITE, exist_ok=True)
for name, data in files.upload().items():
    open(f'{SITE}/{name}', 'wb').write(data)
print(os.listdir(SITE))

In [ ]:
SITES = 'competitor-one competitor-two yoursite'   # every folder you made, same rules for all
!python strip_boilerplate.py scan {SITES}
files.download('boilerplate_lines.csv'); files.download('chrome_entities.csv')

In [ ]:
print('Upload the two review files after you have set each decision to remove / keep')
files.upload()
!python strip_boilerplate.py apply {SITES}

---
## Practical 5.2 — meta descriptions with Claude

Upload `page_entity_profiles.csv` (Entity Extractor) and `pages.csv` (columns: url, title, meta_description, main_content).

In [ ]:
files.upload()
!python meta_descriptions.py --site "Your Site" --limit 5
files.download('review.csv')

Read all five rows. If they are good, run the rest:

In [ ]:
!python meta_descriptions.py --site "Your Site"
files.download('review.csv')

---
## Practical 5.3 — extraction audit (free) and section rewrites (Claude)

Edit the URL list and the main-content selector your site uses.

In [ ]:
open('urls.txt', 'w').write('''https://www.example.com/blog/post-one/
https://www.example.com/blog/post-two/
''')
SELECTOR = 'div.entry-content'
!python extraction_check.py urls.txt --selector "{SELECTOR}"
files.download('extraction_audit.csv')

In [ ]:
# Optional: Claude drafts rewrites of the flagged sections only
!python extraction_check.py urls.txt --selector "{SELECTOR}" --rewrite
files.download('extraction_audit.csv')

---
## Practical 5.5 — entity link plan (free)

Upload `entity_mentions.csv`, `page_entity_profiles.csv`, `naming_table.csv` and Screaming Frog's `all_inlinks.csv`.

In [ ]:
files.upload()
!python link_map.py --min-salience 0.02 --per-page 3
files.download('link_plan.csv'); files.download('link_rules.csv')

---
## Practical 5.6 — JSON-LD from an entity registry (Claude)

Upload your `registry.json` (start from `registry.example.json`) and `pages.csv` (columns: url, title, page_type, main_content).

In [ ]:
files.upload()
!python jsonld_from_registry.py --registry registry.json --pages pages.csv --limit 3
!zip -q -r jsonld.zip jsonld jsonld_review.csv
files.download('jsonld.zip')

---
Need help? The course lessons walk through every step, and the [README](https://github.com/lazarinastoy/mlforseo-code/tree/main/the-ai-powered-entity-audit) has a troubleshooting table.